# pmquant — the one document, cell by cell

This notebook walks `configs/run-e2e.json`: acquired ladders → settlement labels → the banking spine → event panels → a seeded transformer ensemble for q̂ → the D-138 edge gate (owned `validate` + studentized cluster-bootstrap `stat_test`, BH) → the fractional-Kelly MIO over the survivors → the run evaluator.

It runs on the synthetic ladder world the child ships (`pmquant.testing`), acquired through the onboarding platform exactly as real data enters. Run from the child root with the `pmquant` environment (`pip install -e .`).

**Runnable one-liner (same thing, no notebook):**

```bash
python -c "from pmquant.testing import acquire_synthetic; acquire_synthetic('./onboarding_root', {'series': ['KXSYNA','KXSYNB'], 'events_per_series': 80})"
python -m dskit.pipeline run configs/run-e2e.json --asof 2026-04-01 --adapter pmquant
```

## 1. A world, acquired

The synthetic world is BUILT mispriced: every ask sits halfway between the truth and uniform (`shrink=0.5`), a stylized favorite–longshot bias. Acquisition writes WORM snapshots under an onboarding root; the pipeline reads them back through the observations seam, never the generator.

In [ ]:
import json
import tempfile
from pathlib import Path
from pmquant.testing import acquire_synthetic, SyntheticLadderWorld

WORLD = {'seed': 11, 'series': ['KXSYNA', 'KXSYNB'], 'events_per_series': 80, 'rungs': 3, 'start_date': '2026-01-05'}
work = Path(tempfile.mkdtemp(prefix='pmquant-nb-'))
root, registry, source = acquire_synthetic(str(work / 'ob'), WORLD, source_name='synthetic')
world = SyntheticLadderWorld(**WORLD)
print('root:', root.root, '| source alias:', source)
print('events per series:', {s: len(world.events(s)) for s in WORLD['series']})

## 2. The document

The shipped document names the `synthetic` alias; the only edit is `root`. Everything that changes what the run COMPUTES is in the document and graded into its identity hash — `notes` are not.

In [ ]:
from dataclasses import replace
from dskit.pipeline.document import OutputsConfig, load_document
from dskit.pipeline.planner import plan

CHILD = Path.cwd() if (Path.cwd() / 'configs' / 'run-e2e.json').exists() else Path.cwd().parent
raw = json.loads((CHILD / 'configs' / 'run-e2e.json').read_text())
for key in ('ladder_records', 'settlements'):
    raw['pipeline'][key]['params']['root'] = root.root
doc_path = work / 'run-e2e.json'
doc_path.write_text(json.dumps(raw, indent=2))
document = replace(load_document(str(doc_path)), outputs=OutputsConfig(run_root=str(work / 'runs')))
the_plan = plan(document)
print('identity hash:', document.hash[:16])
print('nodes in execution order:', [n.key for n in the_plan.order] if hasattr(the_plan, 'order') else len(raw['pipeline']))

## 3. Run it

One call. Exit-code semantics: `ran` (0), `halted` at a NO-GO gate (3 — a halt is a result), `error` (1).

In [ ]:
import logging
from dskit.pipeline.driver import run_document

logging.basicConfig(level=logging.INFO, format='%(name)s: %(message)s')
result = run_document(document, asof='2026-04-01')
print('state:', result.state, '| run dir:', result.run_dir)
assert result.state == 'ran', result.error

## 4. The banking spine and the panels

Which series banked enough settled events before the train boundary to be tested at all.

In [ ]:
out = result.outputs
print('bank counts:', out['bank']['counts'])
print('eligible:', out['eligible_family'])
print('panels:', out['panels']['metrics'])

## 5. The transformer

Two seeds of the same recipe (AdamW — the pack's default SGD never leaves the uniform prediction here); the checkpoint restored is the epoch with the best per-event validation log-loss over the eligible events.

In [ ]:
for seed in ('seed_0', 'seed_1'):
    m = out[seed]['metrics']
    print(seed, {k: m[k] for k in ('epochs_run', 'selected_epoch', 'monitor_value', 'final_train_loss')})
print('ensemble (cal):', out['ens_cal']['metrics'], '| ensemble (test):', out['ens_test']['metrics'])

## 6. The D-138 edge gate

`validate` scores the CAL band per series, per event: market log-loss minus model log-loss. `stat_test` bootstraps each series' mean delta (studentized, event clusters) and corrects across the family with BH — every series is its own hypothesis; nothing is pooled.

In [ ]:
v = out['validate']['metrics']
print('model log-loss', round(v['loss'], 4), 'vs market', round(v['baseline_loss'], 4), '| beats baseline:', v['beats_baseline'])
edge = out['edge_test']
print('p-values:', edge['pvalues'])
print('verdict:', edge['verdict'], '| survivors:', edge['survivors'])

## 7. The MIO

Per event on the TEST block: the exact-log fractional-Kelly tangent MILP (HiGHS, deterministic) over the entry-gated survivors — fee-aware net edge, depth-walked fills, integer lots, a budget of `deploy_frac × bankroll`.

In [ ]:
s = out['size']
print('lots:', s['lots'], '| outlay:', round(s['outlay'], 2), '| totals:', s['evidence']['totals'])
print('first positions:', dict(list(s['positions'].items())[:6]))
print('per-series:', s['evidence']['instruments'])

## 8. The record

The run evaluator writes `evidence.json` + `evidence.md`; the driver writes `report.md`. Every execute lands in `docs/decisioning/actions.csv` when run from the CLI (pytest runs do not record).

In [ ]:
print(out['run_report']['summary'])
print((Path(result.run_dir) / 'artifacts' / 'run_report' / 'evidence.md').read_text()[:3000])